# 02 One hospital, one helper

**Goal:** train a small "helper" at each hospital using only that hospital's patients, then test every helper on all four hospitals.
You will fill in a 4 by 4 grid and see why a helper trained in one place struggles in another.

**Time:** about 2 hours. Learn 20 min, build 60, explain 15, log 5.

Run each cell from top to bottom. Nothing here needs installing.

## Before you run anything: make a guess

Write your answers in your Results Log or on paper. There are no wrong guesses.

1. Which hospital's helper will do best on its own patients?
2. Which hospital's helper will be worst on the others?
3. Will a helper trained at one hospital do better or worse at a different hospital?

## Step 1: Get the data

If you ran notebook 01 in this same Colab session, the data is already here and this cell will skip the download.
Colab forgets everything when it disconnects, so this cell is safe to run again.

In [ ]:
import os

os.makedirs('data/raw', exist_ok=True)
if not os.path.exists('data/raw/processed.cleveland.data'):
    !wget -q -O data/raw/heart.zip https://archive.ics.uci.edu/static/public/45/heart+disease.zip
    !unzip -o -q data/raw/heart.zip -d data/raw
print(sorted(f for f in os.listdir('data/raw') if f.startswith('processed')))

## Step 2: Load the four hospitals and tidy them

Two small clean-ups, both about **missing values**:

- A `?` in the file means "nobody wrote this down", so we mark it as missing.
- A cholesterol or blood pressure of exactly `0` is not possible for a living person, so it also means "not recorded".

We also turn the original 0 to 4 disease score into a simple yes or no: **sick** (1) or **not sick** (0).

In [ ]:
import numpy as np
import pandas as pd

columns = ['age', 'sex', 'chest_pain', 'resting_bp', 'cholesterol', 'fasting_sugar',
           'rest_ecg', 'max_heart_rate', 'exercise_angina', 'st_depression',
           'st_slope', 'major_vessels', 'thal', 'disease_level']

files = {
    'Cleveland': 'processed.cleveland.data',
    'Hungary': 'processed.hungarian.data',
    'Switzerland': 'processed.switzerland.data',
    'VA Long Beach': 'processed.va.data',
}

hospitals = {}
for name, file in files.items():
    df = pd.read_csv('data/raw/' + file, header=None, names=columns, na_values='?')
    df['cholesterol'] = df['cholesterol'].replace(0, np.nan)
    df['resting_bp'] = df['resting_bp'].replace(0, np.nan)
    df['sick'] = (df['disease_level'] > 0).astype(int)
    hospitals[name] = df

summary = pd.DataFrame({
    'patients': {n: len(d) for n, d in hospitals.items()},
    'share sick': {n: round(d['sick'].mean(), 2) for n, d in hospitals.items()},
    'share of data missing': {n: round(d.drop(columns=['disease_level', 'sick']).isna().mean().mean(), 2)
                              for n, d in hospitals.items()},
})
summary

**Look at the table.** Which hospital has the most missing information? Which has the highest share of sick patients?
Keep this in mind for the grid.

## Step 3: Practice patients and test patients

This is the most important idea of the day.

If you grade a student on the exact questions they practised, you learn nothing about whether they really understand.
So at each hospital we put **70% of patients in a practice pile** (the helper learns from these) and keep **30% hidden as a test pile**
(the helper has never seen these). Grades only count on the test pile.

In [ ]:
from sklearn.model_selection import train_test_split

features = [c for c in columns if c != 'disease_level']

practice, test = {}, {}
for name, df in hospitals.items():
    X_practice, X_test, y_practice, y_test = train_test_split(
        df[features], df['sick'], test_size=0.30, stratify=df['sick'], random_state=42)
    practice[name] = (X_practice, y_practice)
    test[name] = (X_test, y_test)

pd.DataFrame({
    'practice patients': {n: len(practice[n][1]) for n in hospitals},
    'test patients': {n: len(test[n][1]) for n in hospitals},
})

## Step 4: Train one helper per hospital

Each helper is a simple model called **logistic regression**. Think of it as a weighted checklist:
it learns how much each fact (age, blood pressure, chest pain type and so on) should count towards "sick".

Each helper is trained **only on its own hospital's practice pile**.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

helpers = {}
for name in hospitals:
    X_practice, y_practice = practice[name]
    helper = make_pipeline(
        SimpleImputer(strategy='median'),   # fill gaps with the middle value from this hospital
        StandardScaler(),                   # put every fact on a comparable scale
        LogisticRegression(max_iter=1000),
    )
    helper.fit(X_practice, y_practice)
    helpers[name] = helper
    print('trained helper at', name, 'on', len(y_practice), 'practice patients')

## Step 5: Test every helper on every hospital

The result is a 4 by 4 grid. **Rows** are where the helper was trained. **Columns** are where it was tested.
The "home" cells (the diagonal) are a helper tested at its own hospital.

In [ ]:
from sklearn.metrics import accuracy_score

names = list(hospitals)
grid = pd.DataFrame(index=names, columns=names, dtype=float)
for trained_at in names:
    for tested_at in names:
        X_test, y_test = test[tested_at]
        grid.loc[trained_at, tested_at] = accuracy_score(y_test, helpers[trained_at].predict(X_test))

grid.index.name = 'trained at  /  tested at'
grid.round(2)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(grid.values, cmap='YlGnBu', vmin=0.3, vmax=1.0)
ax.set_xticks(range(4)); ax.set_xticklabels(names, rotation=20)
ax.set_yticks(range(4)); ax.set_yticklabels(names)
ax.set_xlabel('Tested at')
ax.set_ylabel('Trained at')
ax.set_title('How often each helper is right')
for i in range(4):
    for j in range(4):
        ax.text(j, i, f'{grid.values[i, j]:.0%}', ha='center', va='center',
                color='white' if grid.values[i, j] > 0.75 else 'black', fontweight='bold' if i == j else None)
plt.colorbar(im, label='share of patients predicted correctly')
plt.tight_layout()
plt.show()

## Step 6: A reality check on what "accuracy" means

At Switzerland, about 93% of patients are sick. A lazy helper that **always says "sick"** would score about 93% without learning anything.
So a high accuracy only means something if it beats the lazy guess.

The table below shows the lazy score for each hospital's test pile. Compare it with your grid.

In [ ]:
lazy = {}
for name in names:
    _, y_test = test[name]
    lazy[name] = round(max(y_test.mean(), 1 - y_test.mean()), 2)

pd.DataFrame({'lazy guess score (always pick the more common answer)': lazy})

## Step 7: Save your results and write the log row

The next cell saves the grid and prints a line you can paste straight into your Results Log.

In [ ]:
os.makedirs('results', exist_ok=True)
grid.round(3).to_csv('results/02_local_models_grid.csv')

home = {n: grid.loc[n, n] for n in names}
away = {n: grid.loc[n, [m for m in names if m != n]].mean() for n in names}
best_home = max(home, key=home.get)
worst_away = min(away, key=away.get)

print('Copy this into your Results Log:\n')
print('Date: 18 Oct 2026')
print('What we tried: Trained one logistic regression helper per hospital, tested each on all four hospitals')
print('Result: home accuracy ->', {n: f'{v:.0%}' for n, v in home.items()})
print('        away accuracy (average on the other three) ->', {n: f'{v:.0%}' for n, v in away.items()})
print('What we learned: best at home =', best_home, '; weakest away =', worst_away)

## Step 8: Explain it to each other (15 minutes)

Do not look at the notebook. Take turns answering out loud, in plain words:

1. Why do we hide a test pile from the helper?
2. Which helper did best at home, and was it just guessing "sick"? (Look at the lazy score.)
3. Pick a helper that did worse away from home. Why might that happen? Think about Day 1: the hospitals have different kinds of patients.
4. If you were a hospital director, what would you want instead of a helper trained only on your own patients?

**Tomorrow (Day 3):** we pool all four hospitals together to find the best case, the ceiling. It would be illegal in real life, but it shows us what we are chasing.